# Football Transfer Value: Model Training Pipeline
This notebook demonstrates the end-to-end process of training our machine learning model. It covers loading the datasets, engineering time-aware features, training the `HistGradientBoostingRegressor`, and evaluating its performance.

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = os.path.dirname(os.path.abspath(''))
DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Data Loading
We load multiple datasets from Transfermarkt (players, appearances, valuations, clubs) and the merged FIFA data.

In [3]:
print('Loading datasets...')
players_df = pd.read_csv(os.path.join(DATA_DIR, 'players_with_fifa.csv'), low_memory=False)
appearances_df = pd.read_csv(os.path.join(DATA_DIR, 'appearances.csv'), low_memory=False)
player_valuations_df = pd.read_csv(os.path.join(DATA_DIR, 'player_valuations.csv'), low_memory=False)
clubs_df = pd.read_csv(os.path.join(DATA_DIR, 'clubs.csv'), low_memory=False)
competitions_df = pd.read_csv(os.path.join(DATA_DIR, 'competitions.csv'), low_memory=False)
print('Data loaded successfully!')

Loading datasets...
Data loaded successfully!


## 2. Feature Engineering
This is the most critical step. We engineer features exactly at the point of the 'valuation date' to prevent data leakage.
- **Demographics:** Age at valuation, years left on contract.
- **Performance:** Career and recent stats normalized per 90 minutes.
- **Advanced Metrics:** `peak_age_decay` and `age_contract_fragility`.
- **FIFA Integration:** Merging scout ratings (overall, potential, reputation).

In [4]:
print('Parsing dates and extracting targets...')
player_valuations_df['date'] = pd.to_datetime(player_valuations_df['date'])
appearances_df['date'] = pd.to_datetime(appearances_df['date'])
players_df['date_of_birth'] = pd.to_datetime(players_df['date_of_birth'], errors='coerce')

player_valuations_df = player_valuations_df.sort_values(by=['player_id', 'date'])
player_valuations_df['previous_market_value'] = player_valuations_df.groupby('player_id')['market_value_in_eur'].shift(1)

target_df = player_valuations_df.groupby('player_id').last().reset_index()
target_df = target_df.rename(columns={'date': 'valuation_date', 'market_value_in_eur': 'target_market_value', 'current_club_id': 'club_id_at_valuation'})
target_df['previous_market_value'] = target_df['previous_market_value'].fillna(100000)

df = players_df.merge(target_df[['player_id', 'valuation_date', 'target_market_value', 'previous_market_value', 'club_id_at_valuation']], on='player_id', how='inner')

# Demographics
df['age_at_valuation'] = (df['valuation_date'] - df['date_of_birth']).dt.days / 365.25
df['age_at_valuation'] = df['age_at_valuation'].fillna(df['age_at_valuation'].median())
df['valuation_year'] = df['valuation_date'].dt.year
df['contract_expiration_date'] = pd.to_datetime(df['contract_expiration_date'], errors='coerce')
df['years_left_on_contract_at_valuation'] = (df['contract_expiration_date'] - df['valuation_date']).dt.days / 365.25
df['years_left_on_contract_at_valuation'] = df['years_left_on_contract_at_valuation'].apply(lambda x: max(0, x) if pd.notnull(x) else 0)

# Time-Aware Performance
apps_merged = appearances_df.merge(target_df[['player_id', 'valuation_date']], on='player_id', how='inner')
career_apps = apps_merged[apps_merged['date'] < apps_merged['valuation_date']]
career_agg = career_apps.groupby('player_id').agg(
    career_appearances=('appearance_id', 'count'), career_goals=('goals', 'sum'),
    career_assists=('assists', 'sum'), career_minutes=('minutes_played', 'sum'),
    career_yellow_cards=('yellow_cards', 'sum'), career_red_cards=('red_cards', 'sum')
).reset_index()

recent_apps = career_apps[(career_apps['valuation_date'] - career_apps['date']).dt.days <= 365]
recent_agg = recent_apps.groupby('player_id').agg(
    recent_appearances=('appearance_id', 'count'), recent_goals=('goals', 'sum'),
    recent_assists=('assists', 'sum'), recent_minutes=('minutes_played', 'sum')
).reset_index()

df = df.merge(career_agg, on='player_id', how='left').merge(recent_agg, on='player_id', how='left')
df.fillna({'career_appearances':0, 'career_goals':0, 'career_assists':0, 'career_minutes':0, 'career_yellow_cards':0, 'career_red_cards':0, 'recent_appearances':0, 'recent_goals':0, 'recent_assists':0, 'recent_minutes':0}, inplace=True)

# Club Data
clubs_comp = clubs_df[['club_id', 'domestic_competition_id', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_seats']]
clubs_comp = clubs_comp.merge(competitions_df[['competition_id', 'sub_type']], left_on='domestic_competition_id', right_on='competition_id', how='left')
clubs_comp.rename(columns={'sub_type': 'competition_tier'}, inplace=True)
df = df.merge(clubs_comp, left_on='club_id_at_valuation', right_on='club_id', how='left')

# Advanced & FIFA Features
df['height_in_cm'] = df['height_in_cm'].fillna(df['height_in_cm'].median())
df['career_goals_per_90'] = (df['career_goals'] / (df['career_minutes'] + 1)) * 90
df['recent_goals_per_90'] = (df['recent_goals'] / (df['recent_minutes'] + 1)) * 90
df['is_veteran'] = (df['age_at_valuation'] >= 34).astype(int)
df['peak_age_decay'] = df['age_at_valuation'].apply(lambda age: max(0, age - 31))
df['age_contract_fragility'] = df['peak_age_decay'] / (df['years_left_on_contract_at_valuation'] + 0.1)

df['fifa_reputation'] = df['fifa_reputation'].fillna(1.0)
df['fifa_overall'] = df['fifa_overall'].fillna(df['fifa_overall'].median())
df['fifa_potential'] = df['fifa_potential'].fillna(df['fifa_overall'])
df['potential_growth'] = df['fifa_potential'] - df['fifa_overall']

print(f'Feature engineering completed! Final dataset shape: {df.shape}')

Parsing dates and extracting targets...
Feature engineering completed! Final dataset shape: (32618, 60)


## 3. Modeling & Evaluation
We use `HistGradientBoostingRegressor` to handle non-linearities and missing values natively.
The target is log-transformed to stabilize the variance of transfer prices.

In [5]:
cat_cols = ['sub_position', 'position', 'foot', 'country_of_citizenship', 'competition_tier']
num_cols = ['height_in_cm', 'age_at_valuation', 'valuation_year', 'years_left_on_contract_at_valuation',
            'career_appearances', 'career_goals', 'career_assists', 'career_minutes', 'career_yellow_cards', 'career_red_cards', 
            'recent_appearances', 'recent_goals', 'recent_assists', 'recent_minutes',
            'career_goals_per_90', 'recent_goals_per_90', 'is_veteran', 'peak_age_decay', 'age_contract_fragility',
            'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_seats',
            'previous_market_value', 'fifa_reputation', 'potential_growth']

final_df = df[num_cols + cat_cols].copy()
for col in cat_cols: final_df[col] = final_df[col].fillna('Unknown')
for col in num_cols: final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(final_df[col].median())

final_df = pd.get_dummies(final_df, drop_first=True, columns=cat_cols)

X = final_df
y = np.log1p(df['target_market_value'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training Model...')
model = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.04, max_depth=12, random_state=42)
model.fit(X_train, y_train)

y_pred = np.expm1(model.predict(X_test))
y_test_real = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
r2 = r2_score(y_test_real, y_pred)

print(f'Test RMSE: {rmse:,.0f} EUR')
print(f'Test R-squared: {r2:.4f}')

# Example Validation
players_to_check = ['Arda Güler', 'Barış Alper Yılmaz', 'Mauro Icardi']
for player_name in players_to_check:
    player_data = df[df['name'].str.contains(player_name, na=False, case=False)]
    if not player_data.empty:
        idx = player_data.index[0]
        actual = player_data.iloc[0]['target_market_value']
        pred = np.expm1(model.predict(X.loc[[idx]]))[0]
        print(f'Validation ({player_name}) | Actual: {actual:,.0f} EUR | Predicted: {pred:,.0f} EUR')

Training Model...
Test RMSE: 1,103,759 EUR
Test R-squared: 0.9607
Validation (Arda Güler) | Actual: 90,000,000 EUR | Predicted: 66,506,044 EUR
Validation (Barış Alper Yılmaz) | Actual: 26,000,000 EUR | Predicted: 24,992,007 EUR
Validation (Mauro Icardi) | Actual: 5,000,000 EUR | Predicted: 5,190,427 EUR
